In [1]:
# 📂 Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
%cd /content/drive/MyDrive

import os

if not os.path.exists("ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git

%cd /content/drive/MyDrive/ComfyUI

!git pull

# 🔥 IMPORTANT: replace PyTorch with CUDA 12.6 build
!pip uninstall -y torch torchvision torchaudio

!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

# 📦 Install other deps AFTER torch
!pip install -r requirements.txt
!pip install rembg onnxruntime

# 🔍 GPU check
import torch
print("✅ CUDA available:", torch.cuda.is_available())
print("💡 GPU device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("🔥 Torch CUDA:", torch.version.cuda)


/content/drive/MyDrive
/content/drive/MyDrive/ComfyUI
Already up to date.
Found existing installation: torch 2.10.0+cu126
Uninstalling torch-2.10.0+cu126:
  Successfully uninstalled torch-2.10.0+cu126
Found existing installation: torchvision 0.25.0+cu126
Uninstalling torchvision-0.25.0+cu126:
  Successfully uninstalled torchvision-0.25.0+cu126
Found existing installation: torchaudio 2.10.0+cu126
Uninstalling torchaudio-2.10.0+cu126:
  Successfully uninstalled torchaudio-2.10.0+cu126
Looking in indexes: https://download.pytorch.org/whl/cu126
  Using cached https://download.pytorch.org/whl/cu126/torch-2.10.0%2Bcu126-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (30 kB)
  Using cached https://download-r2.pytorch.org/whl/cu126/torchvision-0.25.0%2Bcu126-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.4 kB)
  Using cached https://download-r2.pytorch.org/whl/cu126/torchaudio-2.10.0%2Bcu126-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.9 kB)
Using cached https://download.pytorch.org/wh

In [2]:
# Clean up
!rm -rf zrok*

# Download (explicit version to avoid broken "latest")
!wget https://github.com/openziti/zrok/releases/download/v1.1.11/zrok_1.1.11_linux_amd64.tar.gz

# Extract
!tar -xvzf zrok_1.1.11_linux_amd64.tar.gz

# Move binary to working dir
!mv zrok_1.1.11_linux_amd64/zrok ./zrok

# Make executable
!chmod +x zrok

# Test
!./zrok version

# Enable your account (replace with your token)
!./zrok enable THRM7sUcoXW3


--2026-03-19 22:55:13--  https://github.com/openziti/zrok/releases/download/v1.1.11/zrok_1.1.11_linux_amd64.tar.gz
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/515311500/87d07852-1f1e-4cf0-ac01-3d7ecd629ce1?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-19T23%3A46%3A34Z&rscd=attachment%3B+filename%3Dzrok_1.1.11_linux_amd64.tar.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-19T22%3A46%3A10Z&ske=2026-03-19T23%3A46%3A34Z&sks=b&skv=2018-11-09&sig=huPjNpd%2BlSHoglw6k0%2FJwGB8e14TNHEmPE4X3YbH3Gg%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3Mzk2MjcxMywibmJmIjoxNzczOTYwOTEzLCJwYXRoIjoicmVsZ

In [2]:
!mkdir -p /content/drive/MyDrive/zrok
!cp -r ~/.zrok /content/drive/MyDrive/zrok

In [3]:
# ==========================================
# Hunyuan3D 2.1 Custom Node
# ==========================================

import os
import sys
import subprocess

COMFY_PATH = "/content/drive/MyDrive/ComfyUI"

CUSTOM_NODES = f"{COMFY_PATH}/custom_nodes"
NODE_PATH = f"{CUSTOM_NODES}/ComfyUI-Hunyuan3d-2-1"

print("🚀 Setting up Hunyuan3D 2.1...")

print("🔧 Installing system dependencies...")
subprocess.run(["apt-get", "update"])
subprocess.run(["apt-get", "install", "-y", "build-essential"])


if not os.path.exists(NODE_PATH):
    print("📦 Cloning repository...")
    subprocess.run([
        "git", "clone",
        "https://github.com/visualbruno/ComfyUI-Hunyuan3d-2-1",
        NODE_PATH
    ])
else:
    print("✅ Repository already exists")

print("📦 Installing Python dependencies...")
req_path = os.path.join(NODE_PATH, "requirements.txt")
if os.path.exists(req_path):
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", req_path])
else:
    print("⚠️ No requirements.txt found")


print("🧠 Installing custom rasterizer...")

rasterizer_dir = f"{NODE_PATH}/hy3dpaint/custom_rasterizer"
rasterizer_dist = f"{rasterizer_dir}/dist"

installed = False

if os.path.exists(rasterizer_dist):
    for file in os.listdir(rasterizer_dist):
        if "linux" in file and file.endswith(".whl"):
            wheel_path = os.path.join(rasterizer_dist, file)
            print(f"⚡ Trying wheel: {file}")
            result = subprocess.run([sys.executable, "-m", "pip", "install", wheel_path])
            if result.returncode == 0:
                installed = True
                break

if not installed:
    print("🔧 Falling back to source build (rasterizer)...")
    subprocess.run([sys.executable, "-m", "pip", "install", rasterizer_dir])

print("🎨 Installing differentiable renderer...")

renderer_dir = f"{NODE_PATH}/hy3dpaint/DifferentiableRenderer"
renderer_dist = f"{renderer_dir}/dist"

installed = False

if os.path.exists(renderer_dist):
    for file in os.listdir(renderer_dist):
        if "linux" in file and file.endswith(".whl"):
            wheel_path = os.path.join(renderer_dist, file)
            print(f"⚡ Trying wheel: {file}")
            result = subprocess.run([sys.executable, "-m", "pip", "install", wheel_path])
            if result.returncode == 0:
                installed = True
                break

if not installed:
    print("🔧 Falling back to source build (renderer)...")
    subprocess.run([sys.executable, "-m", "pip", "install", renderer_dir])



🚀 Setting up Hunyuan3D 2.1...
🔧 Installing system dependencies...
✅ Repository already exists
📦 Installing Python dependencies...
🧠 Installing custom rasterizer...
⚡ Trying wheel: custom_rasterizer-0.1-cp311-cp311-linux_x86_64.whl
🔧 Falling back to source build (rasterizer)...
🎨 Installing differentiable renderer...
⚡ Trying wheel: mesh_inpaint_processor-0.0.0-cp311-cp311-linux_x86_64.whl
🔧 Falling back to source build (renderer)...


In [20]:
!pkill -f main.py

In [23]:
import subprocess
import time

comfy_process = subprocess.Popen(
    [
        "python",
        "main.py",
        "--listen", "0.0.0.0",
        "--port", "8188",
        "--disable-smart-memory",
        "--force-fp32"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)


# Wait until server is ready
ready = False
for _ in range(60):
    line = comfy_process.stdout.readline()
    if line:
        print(line.strip())

        if "Starting server" in line or "To see the GUI go to" in line:
            ready = True
            break

    time.sleep(1)

if ready:
    print("✅ ComfyUI is running on port 8188")
else:
    print("❌ ComfyUI may not have started correctly")



Found comfy_kitchen backend triton: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['apply_rope', 'apply_rope1', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'quantize_mxfp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8']}
Found comfy_kitchen backend eager: {'available': True, 'disabled': False, 'unavailable_reason': None, 'capabilities': ['apply_rope', 'apply_rope1', 'dequantize_mxfp8', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'quantize_mxfp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8', 'scaled_mm_mxfp8', 'scaled_mm_nvfp4']}
Found comfy_kitchen backend cuda: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['apply_rope', 'apply_rope1', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'quantize_mxfp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8', 'scaled_mm_nvfp4']}
Checkpoint files will always be loaded safely.
Total VRAM 81153 MB, total RAM 171061 MB
pytorch version: 2.10.0+cu126
Forcing FP32, if this im

In [24]:
!./zrok share public http://127.0.0.1:8188


╭──────────────────────────────────╮╭──────────────────────────╮
│https://mzztl3mstnap.share.zrok.io││     [PUBLIC] [PROXY]     │
╰──────────────────────────────────╯╰──────────────────────────╯
╭╮                                                              
││                                                              
╰╯                                                              

In [11]:
from fastapi import FastAPI, HTTPException, Query, Request, Response
import copy
import json
import os
import requests
import time
import uuid

app = FastAPI()

COMFYUI_URL = "http://127.0.0.1:8188"
WORKFLOW_TEMPLATE_PATH = os.path.join(
    "/content/drive/MyDrive",
    "workflows",
    "spatialgen_workflow_api.json"
)



def load_default_workflow():
    if not os.path.exists(WORKFLOW_TEMPLATE_PATH):
        return None

    with open(WORKFLOW_TEMPLATE_PATH, "r", encoding="utf-8") as f:
        return json.load(f)


def apply_prompt_inputs(workflow, prompt, negative_prompt=""):
    for node in workflow.values():
        if not isinstance(node, dict) or node.get("class_type") != "CLIPTextEncode":
            continue

        meta = node.get("_meta") or {}
        title = str(meta.get("title", "")).lower()
        inputs = node.setdefault("inputs", {})

        if "negative" in title:
            inputs["text"] = negative_prompt
        else:
            inputs["text"] = prompt


def apply_generation_inputs(workflow, generation):
    if not generation:
        return

    seed = generation.get("seed")
    steps = generation.get("steps")
    cfg = generation.get("cfg")
    sampler = generation.get("sampler")
    width = generation.get("width")
    height = generation.get("height")

    for node in workflow.values():
        if not isinstance(node, dict):
            continue

        class_type = node.get("class_type")
        inputs = node.setdefault("inputs", {})

        if class_type == "KSampler":
            if seed is not None:
                inputs["seed"] = seed
            if steps is not None:
                inputs["steps"] = steps
            if cfg is not None:
                inputs["cfg"] = cfg
            if sampler:
                inputs["sampler_name"] = sampler
        elif class_type == "EmptyLatentImage":
            if width is not None:
                inputs["width"] = width
            if height is not None:
                inputs["height"] = height


def build_workflow(body):
    workflow = body.get("workflow")
    if workflow is None:
        workflow = load_default_workflow()
        if workflow is None:
            raise HTTPException(
                status_code=400,
                detail=(
                    "No workflow provided. Upload a default template to "
                    f"{WORKFLOW_TEMPLATE_PATH} or send workflow JSON from Unity."
                ),
            )

    if isinstance(workflow, dict) and "nodes" in workflow and "links" in workflow:
        raise HTTPException(
            status_code=400,
            detail=(
                "Workflow is in ComfyUI UI format, not API format. "
                "Open it in ComfyUI and export/save it in API format before using /generate."
            ),
        )

    workflow = copy.deepcopy(workflow)
    apply_prompt_inputs(
        workflow,
        body.get("prompt", ""),
        body.get("negative_prompt", ""),
    )
    apply_generation_inputs(workflow, body.get("generation") or {})
    return workflow


def fetch_history(prompt_id):
    response = requests.get(f"{COMFYUI_URL}/history/{prompt_id}", timeout=30)
    response.raise_for_status()
    return response.json()


def get_history_entry(history_payload, prompt_id):
    if isinstance(history_payload, dict):
        if prompt_id in history_payload:
            return history_payload[prompt_id]
        if "outputs" in history_payload or "status" in history_payload:
            return history_payload
    return None


def extract_execution_error(history_entry):
    if not isinstance(history_entry, dict):
        return ""

    status = history_entry.get("status") or {}
    messages = status.get("messages") or []
    for message in messages:
        if not isinstance(message, list) or len(message) < 2:
            continue
        event_type, payload = message[0], message[1]
        if event_type != "execution_error" or not isinstance(payload, dict):
            continue
        return payload.get("exception_message") or payload.get("exception_type") or ""

    return ""


def extract_images(outputs):
    images = []
    for node_id, node_output in outputs.items():
        if "images" in node_output:
            for img in node_output["images"]:
                images.append({
                    "filename": img["filename"],
                    "subfolder": img.get("subfolder", ""),
                    "type": img.get("type", "output")
                })
    return images


@app.get("/health")
def health():
    try:
        response = requests.get(f"{COMFYUI_URL}/system_stats", timeout=10)
        response.raise_for_status()
        return {"status": "ok", "comfyui": "reachable"}
    except requests.RequestException as exc:
        raise HTTPException(status_code=503, detail=f"ComfyUI unreachable: {exc}") from exc


@app.post("/generate")
async def generate(req: Request):
    body = await req.json()

    request_id = body.get("request_id") or str(uuid.uuid4())
    start_time = time.time()

    try:
        workflow = build_workflow(body)
        response = requests.post(
            f"{COMFYUI_URL}/prompt",
            json={"prompt": workflow},
            timeout=60,
        )

        if not response.ok:
            detail = response.text
            print(f"[{request_id}] ComfyUI /prompt validation failed: {detail}")
            raise HTTPException(
                status_code=502,
                detail=f"ComfyUI /prompt rejected workflow: {detail}",
            )

        result = response.json()
        prompt_id = result.get("prompt_id")
        duration = time.time() - start_time

        if not prompt_id:
            raise HTTPException(status_code=502, detail=f"ComfyUI /prompt missing prompt_id: {result}")

        print(f"[{request_id}] queued prompt_id={prompt_id} ({duration:.2f}s)")

        return {
            "request_id": request_id,
            "status": "queued",
            "duration": duration,
            "prompt_id": prompt_id,
            "comfyui_response": result,
        }

    except HTTPException:
        raise
    except requests.RequestException as exc:
        duration = time.time() - start_time
        response_text = exc.response.text if exc.response is not None else str(exc)
        print(f"[{request_id}] request error: {response_text}")
        raise HTTPException(status_code=502, detail=f"Failed to submit prompt to ComfyUI: {response_text}") from exc
    except Exception as exc:
        duration = time.time() - start_time
        print(f"[{request_id}] unexpected error after {duration:.2f}s: {exc}")
        raise HTTPException(status_code=500, detail=str(exc)) from exc


@app.get("/result/{prompt_id}")
def get_result(prompt_id: str):
    try:
        history_payload = fetch_history(prompt_id)
        history_entry = get_history_entry(history_payload, prompt_id)

        if history_entry is None:
            return {
                "status": "running",
                "prompt_id": prompt_id,
                "completed": False,
                "images": [],
                "history": {},
            }

        outputs = history_entry.get("outputs") or {}
        images = extract_images(outputs)
        completed = bool((history_entry.get("status") or {}).get("completed")) or bool(images)
        error_message = extract_execution_error(history_entry)
        status = "error" if error_message else ("success" if completed else "running")

        return {
            "status": status,
            "prompt_id": prompt_id,
            "completed": completed,
            "exception_message": error_message,
            "images": images,
            "history": {prompt_id: history_entry},
        }
    except requests.RequestException as exc:
        raise HTTPException(status_code=502, detail=f"Failed to query ComfyUI history: {exc}") from exc


@app.get("/history/{prompt_id}")
def proxy_history(prompt_id: str):
    try:
        return fetch_history(prompt_id)
    except requests.RequestException as exc:
        raise HTTPException(status_code=502, detail=f"Failed to query ComfyUI history: {exc}") from exc


@app.get("/view")
def proxy_view(
    filename: str = Query(...),
    subfolder: str = Query(""),
    type: str = Query("output"),
):
    try:
        response = requests.get(
            f"{COMFYUI_URL}/view",
            params={
                "filename": filename,
                "subfolder": subfolder,
                "type": type,
            },
            timeout=60,
        )
        response.raise_for_status()
        return Response(
            content=response.content,
            media_type=response.headers.get("content-type", "application/octet-stream"),
        )
    except requests.RequestException as exc:
        raise HTTPException(status_code=502, detail=f"Failed to fetch ComfyUI output file: {exc}") from exc


In [12]:
import uvicorn
from threading import Thread

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = Thread(target=run)
thread.start()

print("✅ FastAPI proxy running on port 8000")


✅ FastAPI proxy running on port 8000


INFO:     Started server process [6108]


In [14]:
import subprocess
import time
import re

def start_zrok():
    return subprocess.Popen(
        [
            './zrok',
            'share',
            'reserved',
            'comfyuitunnel',
            '--headless'   # ✅ THIS FIXES YOUR ERROR
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

zrok_process = start_zrok()

public_url = None

for _ in range(30):
    line = zrok_process.stdout.readline()
    if line:
        print(line.strip())

        match = re.search(r'https://[^\s]+', line)
        if match:
            public_url = match.group(0)
            print(f"\n🚀 PUBLIC ENDPOINT: {public_url}\n")
            break

    time.sleep(1)

if not public_url:
    print("❌ Failed to start zrok")



{"file":"/__w/zrok/zrok/cmd/zrok/shareReserved.go:132","func":"main.(*shareReservedCommand).shareLocal","level":"info","msg":"sharing target: 'http://localhost:8000'","time":"2026-03-19T22:08:11.190Z"}
{"file":"/__w/zrok/zrok/cmd/zrok/shareReserved.go:147","func":"main.(*shareReservedCommand).shareLocal","level":"info","msg":"using existing backend target: http://localhost:8000","time":"2026-03-19T22:08:11.190Z"}
{"file":"/__w/zrok/zrok/cmd/zrok/shareReserved.go:330","func":"main.(*shareReservedCommand).shareLocal","level":"info","msg":"access your zrok share: https://comfyuitunnel.share.zrok.io","time":"2026-03-19T22:08:12.529Z"}

🚀 PUBLIC ENDPOINT: https://comfyuitunnel.share.zrok.io","time":"2026-03-19T22:08:12.529Z"}

